In [1]:
import time
from oggm import cfg, utils, workflow, tasks, graphics
from oggm.sandbox import ioggm_dynamic_spinup
from oggm.core.sia2d import IGM_Model2D, compute_2d_quantiles # import the IGM_Model2D
from datetime import datetime
import os
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
from oggm.shop import gcm_climate
from oggm.core.massbalance import (DistributedMassBalance,
                                   MonthlyTIModel)
import pandas as pd


/Users/afisc/igm_venv/bin/python


## volume or area minimisation?

In [2]:
# min_for = 'area'
min_for = 'volume'

In [3]:
cfg.initialize(logging_level='WARNING')

ts = datetime.now().strftime("%Y%m%d-%H%M%S")
cfg.PATHS['working_dir'] = os.path.join('/Users/afisc/instructed-oggm/experiments_working_dir/06_11/iOGGM_melt_f_and_gcm/{}__{}'.format(ts, min_for))


2026-06-17 09:21:01: oggm.cfg: Reading default parameters from the OGGM `params.cfg` configuration file.
2026-06-17 09:21:01: oggm.cfg: Multiprocessing switched OFF according to the parameter file.
2026-06-17 09:21:01: oggm.cfg: Multiprocessing: using all available processors (N=8)


## choose the glacier to work with

In [4]:
# Hintereisferner and a 'bad' glacier in terms of the dynamic spinup
# rgi_ids = ['RGI60-11.00897', 'RGI60-11.00275']

## Pick a glacier
rgi_ids = ['RGI60-11.01450']  # This is Aletsch
# rgi_ids = ['RGI60-11.00897']  # This is Hintereisferner
# rgi_ids = ['RGI60-11.03638']  # This is Argentiere

## defining and loading the pre-processed gdir data

In [5]:
# We use a recent gdir setting, calibated on a glacier per glacier basis
base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/'
            'L3-L5_files/2023.3/elev_bands/W5E5/')
# base_url = ('https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L3-L5_files/2023.3/elev_bands/W5E5_spinup')


In [6]:
gdirs = workflow.init_glacier_directories(rgi_ids,from_prepro_level=3, prepro_border=80, prepro_base_url=base_url)



2026-06-17 09:21:02: oggm.workflow: init_glacier_directories from prepro level 3 on 1 glaciers.
2026-06-17 09:21:02: oggm.workflow: Execute entity tasks [gdir_from_prepro] on 1 glaciers


In [7]:
gdir = gdirs[0]

In [8]:
gdir.read_json('mb_calib')

{'rgi_id': 'RGI60-11.01450',
 'bias': 0,
 'melt_f': 6.547706124767281,
 'prcp_fac': 1.4212121577833927,
 'temp_bias': 0.9206985061156576,
 'reference_mb': -1210.7,
 'reference_mb_err': 131.5,
 'reference_period': '2000-01-01_2020-01-01',
 'mb_global_params': {'temp_default_gradient': -0.0065,
  'temp_all_solid': 0.0,
  'temp_all_liq': 2.0,
  'temp_melt': -1.0},
 'baseline_climate_source': 'GSWP3_W5E5'}

In [9]:
# add consensus now as well, as we use a different prepro
from oggm.shop import bedtopo

workflow.execute_entity_task(bedtopo.add_consensus_thickness, gdir);

2026-06-17 09:21:02: oggm.workflow: Execute entity tasks [add_consensus_thickness] on 1 glaciers


In [10]:
thick = tasks.distribute_thickness_per_altitude(gdir)

In [11]:
spinup_start_yr = 1979
output_filesuffix= f'meltf_{min_for}'
kwargs_run_function = {
    'glen_a' : 68.061,
    'slidingco' : 0.068115,
    'store_diagnostics': True,
    'store_all_spinup_steps': True,
    'store_diagnostics_spinup': True,
    'store_model_geometry_spinup': True,
    'store_model_geometry': True,
    'minimise_for': min_for,
    'mb_filter_value': -5
}
tasks.run_dynamic_ioggm_melt_f_calibration(gdir,
                         ys=spinup_start_yr,  # When to start the spinup
                         output_filesuffix=output_filesuffix,  # Where to write the output
                         ye=2020,  # When the simulation should stop
                         kwargs_run_function=kwargs_run_function,
                         # evolution_model=IGM_Model2D # Using the IGM Model
                         );

2026-06-17 09:21:05.345382: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-06-17 09:21:05.345418: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-06-17 09:21:05.345434: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-06-17 09:21:05.345568: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-17 09:21:05.345747: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a


2026-06-17 09:21:05.858433: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
/Users/afisc/igm_venv/lib/python3.10/site-packages/tensorflow/python/framework/tensor_util.py:518: RuntimeWarning: overflow encountered in cast
  nparray = np.array(values, dtype=np_dt)


mb_model_historical temp bias: 0.9206985061156576
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
mb_model_historical temp bias: 0.9206985061156576
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
mb_model_historical temp bias: 0.9206985061156576
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
mb_model_historical temp bias: 0.9206985061156576
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
mb_model_historical temp bias: 0.9206985061156576
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
RAN THE SPINUP: CURRENT DMDTDA_mdl is-203.75715025111626
Found pretrained emulator i

## Load gcm data

In [12]:
temperature_scenarios = False

In [13]:
# load CMIP5 + CMIP6 metadata
gcms_cmip6 = pd.read_csv('/Users/afisc/atmo_master/rgi_job/oggm/cmip_data/cmip6/all_gcm_list.csv', index_col=0)
gcms_cmip5 = pd.read_csv('/Users/afisc/atmo_master/rgi_job/oggm/cmip_data/cmip5-ng/all_gcm_list.csv', index_col=0)

In [14]:
def remote_path(path):
    base_local = "/home/www/oggm/"
    base_remote = "https://cluster.klima.uni-bremen.de/~oggm/"

    if path.startswith(base_local):
        return path.replace(base_local, base_remote, 1)
    else:
        raise ValueError(f"Path does not start with {base_local}")

In [15]:
if not temperature_scenarios:
    # set the ssp scenarios here
    scenarios = ['ssp126', 
                 'ssp245', 
                 'ssp585']


    def get_models_per_scenario(scenario):
        return np.unique(gcms_cmip6[gcms_cmip6.ssp == scenario].gcm.values)


    gcms_per_scenario = {}
    for scenario in scenarios:
        gcms_per_scenario[scenario] = get_models_per_scenario(scenario)
        # if is_notebook:
        #     gcms_per_scenario[scenario] = [gcms_per_scenario[scenario][0]]

    # iterate through the scenarios        
    for scenario in gcms_per_scenario:
        for gcm in gcms_per_scenario[scenario]:
            rid = f'_{gcm}_{scenario}'

            select_gcm = np.array([g.upper() for g in gcms_cmip6.gcm]) == gcm.upper()
            select_ssp = gcms_cmip6.ssp == scenario
            selected_run = gcms_cmip6[select_gcm & select_ssp]
            ft = selected_run[selected_run['var'] == 'tas'].path.values[0]
            fp = selected_run[selected_run['var'] == 'pr'].path.values[0]

            ft = utils.file_downloader(remote_path(ft))
            fp = utils.file_downloader(remote_path(fp))
            print(ft)
            # bias correct them
            workflow.execute_entity_task(gcm_climate.process_cmip_data, gdirs,
                                         year_range=('2000', '2019'),
                                         filesuffix=rid,  # recognize the climate file for later
                                         fpath_temp=ft,  # temperature projections
                                         fpath_precip=fp,  # precip projections
                                         );


2026-06-17 09:35:00: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers


/Users/afisc/OGGM/download_cache/cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/ACCESS-CM2/ACCESS-CM2_ssp126_r1i1p1f1_tas.nc


2026-06-17 09:35:04: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers


/Users/afisc/OGGM/download_cache/cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/ACCESS-ESM1-5/ACCESS-ESM1-5_ssp126_r1i1p1f1_tas.nc


2026-06-17 09:35:08: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers


...

/Users/afisc/OGGM/download_cache/cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/MRI-ESM2-0/MRI-ESM2-0_ssp585_r1i1p1f1_tas.nc


2026-06-17 09:36:20: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers


/Users/afisc/OGGM/download_cache/cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/NorESM2-MM/NorESM2-MM_ssp585_r1i1p1f1_tas.nc


2026-06-17 09:36:21: oggm.workflow: Execute entity tasks [process_cmip_data] on 1 glaciers


/Users/afisc/OGGM/download_cache/cluster.klima.uni-bremen.de/~oggm/cmip6/GCM/TaiESM1/TaiESM1_ssp585_r1i1p1f1_tas.nc


In [16]:
for ssp_scenario, gcms in gcms_per_scenario.items():
    gcms = gcms.astype(str)
    gcms_per_scenario[ssp_scenario] = np.char.add(np.char.add("_", gcms), f"_{ssp_scenario}").tolist()
    scenario_suffixes = gcms_per_scenario


## Load Post-Spinup Geometry


In [17]:
fp = gdir.get_filepath('ioggm_geometry', filesuffix=output_filesuffix)

In [18]:
with xr.open_dataset(fp) as ds:
    post_spinup_geom = ds.load()

post_spinup_thickness = post_spinup_geom['ice_thickness'].isel(time=-1)

## load topography

In [19]:
with xr.open_dataset(gdir.get_filepath('gridded_data')) as gd:
    gd = gd.load()
bed_con = gd.topo - gd.consensus_ice_thickness.fillna(0)

## GCM run 2020-2100


In [20]:
for scenario in scenario_suffixes.values():
    for scenario_suffix in scenario:

        for gdir in gdirs:
            geom_path = gdir.get_filepath('ioggm_geometry',
                                         filesuffix=scenario_suffix,
                                         delete=True)
            
            diag_path = gdir.get_filepath('ioggm_diagnostics',
                                          filesuffix=scenario_suffix,
                                          delete=True)
            
            # initialize mb model
            mb_model = DistributedMassBalance(
                gdir,
                mb_model_class=MonthlyTIModel,
                filename='gcm_data',
                input_filesuffix=scenario_suffix,
                # temp_bias=2,
                # melt_f=10,
            )

            # initialize model
            model = IGM_Model2D(bed_con.values.astype('float32'), init_ice_thick=post_spinup_thickness.fillna(0).values.astype('float32'), 
                                    dx=gdir.grid.dx, dy=gdir.grid.dy, x=bed_con.x, y=bed_con.y, mb_model=mb_model, # distributed_mb_model=True,
                                    y0=2020, mb_filter=gd.glacier_mask.values==1, 
                                    glen_a=68.061, slidingco=0.068115, mb_filter_value=-5)

            # run model
            model.run_until_and_store(2100, geom_path=geom_path, diag_path=diag_path, grid=gdir.grid) 
            print(f'{scenario_suffix} done!')

Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_ACCESS-CM2_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_ACCESS-ESM1-5_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_BCC-CSM2-MR_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_CAMS-CSM1-0_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_CESM2_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_CESM2-WACCM_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_CanESM5_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_EC-Earth3_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_EC-Earth3-Veg_ssp126 done!
Found pretrained emulator in the igm package: pinnbp_10_4_cnn_16_32_2_1_a
_FGOALS-f3-L_ssp126 done!
Found pre

In [21]:
from oggm.core.sia2d import compute_2d_quantiles
for scenario_name, suffix_list in scenario_suffixes.items():
    
    compute_2d_quantiles(gdir, suffix_list, quantiles=[0.5, 0.25, 0.75], output_filesuffix=f'_{scenario_name}_quantiles')
    compute_2d_quantiles(gdir, suffix_list, filename='ioggm_diagnostics', quantiles=[0.5, 0.25, 0.75], output_filesuffix=f'_{scenario_name}_quantiles')